# Pascal VOC YOLOv8n Transfer Learning

This notebook fine-tunes the prepared Pascal VOC dataset and YOLOv8n checkpoint on Kaggle GPU hardware.

In [ ]:
import sys
import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(index, torch.cuda.get_device_name(index))

In [ ]:
!pip install -q --no-deps ultralytics==8.3.0

import ultralytics
print('Ultralytics:', ultralytics.__version__)

In [ ]:
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/voc_project')
WORK_ROOT.mkdir(parents=True, exist_ok=True)

files = [path for path in INPUT_ROOT.rglob('*') if path.is_file()]
print(f'Files found: {len(files)}')
for path in files[:100]:
    print(path)

archives = [path for path in files if path.suffix.lower() == '.zip']
train_dirs = list(INPUT_ROOT.rglob('train/images'))
val_dirs = list(INPUT_ROOT.rglob('val/images'))
test_dirs = list(INPUT_ROOT.rglob('test/images'))

print('ZIP archives:', archives)
print('Existing train directories:', train_dirs)
print('Existing validation directories:', val_dirs)
print('Existing test directories:', test_dirs)

if not archives and not (train_dirs and val_dirs and test_dirs):
    raise FileNotFoundError(
        'Attached input was found, but it contains neither ZIP archives nor '
        'train/images, val/images, and test/images directories.'
    )

In [ ]:
for archive in archives:
    print('Extracting:', archive.name)
    with zipfile.ZipFile(archive) as zip_file:
        zip_file.extractall(WORK_ROOT)
print('Extraction complete.')

In [ ]:
from pathlib import Path
import zipfile

# Extract attached archives if the working directory does not contain the splits yet.
WORK_ROOT.mkdir(parents=True, exist_ok=True)
existing_train = list(WORK_ROOT.rglob('train/images'))
if not existing_train:
    archives = list(INPUT_ROOT.rglob('*.zip'))
    print('Archives to extract:', archives)
    for archive in archives:
        with zipfile.ZipFile(archive) as zip_file:
            zip_file.extractall(WORK_ROOT)

# Search both the extracted working directory and Kaggle input.
search_roots = [WORK_ROOT, INPUT_ROOT]
train_dirs = []
val_dirs = []
test_dirs = []
for root in search_roots:
    train_dirs.extend(root.rglob('train/images'))
    val_dirs.extend(root.rglob('val/images'))
    test_dirs.extend(root.rglob('test/images'))

print('Train:', train_dirs)
print('Validation:', val_dirs)
print('Test:', test_dirs)

if not train_dirs or not val_dirs or not test_dirs:
    raise FileNotFoundError(
        'Train, validation, or test images were not found. '
        'Run the input diagnostic cell and confirm the Kaggle dataset is attached.'
    )

train_dir = train_dirs[0]
val_dir = val_dirs[0]
test_dir = test_dirs[0]
DATA_ROOT = train_dir.parent.parent

print('Dataset root:', DATA_ROOT)
print('Train images:', len(list(train_dir.glob('*.jpg'))))
print('Validation images:', len(list(val_dir.glob('*.jpg'))))
print('Test images:', len(list(test_dir.glob('*.jpg'))))

In [ ]:
checkpoint_roots = [WORK_ROOT, INPUT_ROOT]
checkpoint_files = []
for root in checkpoint_roots:
    checkpoint_files.extend(root.rglob('*.pt'))

print('Checkpoints:', checkpoint_files)
last_files = [path for path in checkpoint_files if path.name == 'last.pt']
best_files = [path for path in checkpoint_files if path.name == 'best.pt']
CHECKPOINT = last_files[0] if last_files else best_files[0] if best_files else None

if CHECKPOINT is None:
    raise FileNotFoundError(
        'No best.pt or last.pt checkpoint was found in Kaggle input or working storage.'
    )

print('Using checkpoint:', CHECKPOINT)

In [ ]:
import yaml

DATA_YAML = Path('/kaggle/working/voc_kaggle.yaml')

# The train, validation, and test archives are separate Kaggle inputs.
# Use absolute paths so Ultralytics does not assume one shared dataset root.
config = {
    'path': '/',
    'train': str(train_dir),
    'val': str(val_dir),
    'test': str(test_dir),
    'names': [
        'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
        'bus', 'car', 'cat', 'chair', 'cow',
        'diningtable', 'dog', 'horse', 'motorbike', 'person',
        'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
    ]
}

DATA_YAML.write_text(yaml.safe_dump(config, sort_keys=False))
print(DATA_YAML.read_text())

In [ ]:
from ultralytics.data.utils import check_det_dataset
dataset_info = check_det_dataset(str(DATA_YAML))
print('Train:', dataset_info['train'])
print('Validation:', dataset_info['val'])
print('Test:', dataset_info['test'])
print('Dataset validation passed.')

## Fine-tuning

`resume=False` is intentional because the local checkpoint contains the old local dataset path. The checkpoint is used as model initialization with the corrected Kaggle YAML.

In [ ]:
import os
os.environ['WANDB_DISABLED'] = 'true'
os.environ['WANDB_MODE'] = 'disabled'

import wandb
wandb.init(mode='disabled')

# Kaggle's preinstalled ray version is missing an attribute the Ultralytics
# raytune callback expects; patch it since Ray Tune is not being used here.
import ray.train._internal.session as ray_session
if not hasattr(ray_session, '_get_session'):
    ray_session._get_session = lambda: None

from ultralytics import YOLO

model = YOLO(str(CHECKPOINT))
model.train(
    data=str(DATA_YAML),
    epochs=30,
    resume=False,
    imgsz=640,
    batch=8,
    workers=2,
    device=0,
    patience=8,
    amp=True,
    save_period=5,
    project='/kaggle/working/runs/detect',
    name='voc_yolov8n_t4',
    seed=42,
    deterministic=True,
    cache=False
)

## Final test evaluation

Run only after training completes. Uses the untouched test split; results are not used to modify the model.

In [ ]:
detect_root = Path('/kaggle/working/runs/detect')
run_dirs = sorted(detect_root.glob('voc_yolov8n_t4*'), key=lambda p: p.stat().st_mtime)
if not run_dirs:
    raise FileNotFoundError(f'No training run directories found under {detect_root}.')
BEST_TRAINED = run_dirs[-1] / 'weights' / 'best.pt'
print('Using best checkpoint:', BEST_TRAINED)

best_model = YOLO(str(BEST_TRAINED))

test_metrics = best_model.val(
    data=str(DATA_YAML),
    split='test',
    imgsz=640,
    batch=8,
    device=0,
    conf=0.001,
    iou=0.7,
    project='/kaggle/working/runs/eval',
    name='voc_test',
    plots=True
)

## Sample predictions

Generates annotated prediction images on the test split for the report.

In [ ]:
best_model.predict(
    source=str(test_dir),
    imgsz=640,
    batch=8,
    device=0,
    conf=0.25,
    iou=0.7,
    save=True,
    project='/kaggle/working/runs/predict',
    name='voc_test_predictions'
)

In [ ]:
import shutil
from pathlib import Path

EVAL_DIR = Path('/kaggle/working/runs/eval')
ZIP_PATH = Path('/kaggle/working/voc_eval_results')

if not EVAL_DIR.exists():
    raise FileNotFoundError(f'{EVAL_DIR} does not exist. Run test evaluation first.')

archive_path = shutil.make_archive(str(ZIP_PATH), 'zip', root_dir=str(EVAL_DIR))
print('Created:', archive_path)
print('Size (MB):', round(Path(archive_path).stat().st_size / (1024 * 1024), 2))

In [ ]:
import shutil
from pathlib import Path

PREDICT_DIR = Path('/kaggle/working/runs/predict')
ZIP_PATH = Path('/kaggle/working/voc_predictions')

if not PREDICT_DIR.exists():
    raise FileNotFoundError(f'{PREDICT_DIR} does not exist. Run prediction generation first.')

archive_path = shutil.make_archive(str(ZIP_PATH), 'zip', root_dir=str(PREDICT_DIR))
print('Created:', archive_path)
print('Size (MB):', round(Path(archive_path).stat().st_size / (1024 * 1024), 2))